In [2]:
use std::fs::File;
use std::io::{Read, Seek, SeekFrom};

// ===== CRC32テーブル (コンパイル時定数) =====
const CRC_TABLE: [u32; 256] = {
    let mut table = [0u32; 256];
    let mut i = 0;
    while i < 256 {
        let mut c = i as u32;
        let mut j = 0;
        while j < 8 {
            c = if c & 1 != 0 { (c >> 1) ^ 0xedb88320 } else { c >> 1 };
            j += 1;
        }
        table[i] = c;
        i += 1;
    }
    table
};

// ===== ヘルパー関数 =====
#[inline(always)]
fn crc32(c: u32, d: u8) -> u32 {
    (c >> 8) ^ CRC_TABLE[((c ^ d as u32) & 0xff) as usize]
}

#[inline(always)]
fn decrypt_byte(key2: u32) -> u8 {
    let temp = (key2 | 2) as u16;
    ((u32::from(temp) * u32::from(temp ^ 1)) >> 8) as u8
}

#[inline(always)]
fn msb(x: u32) -> u8 { (x >> 24) as u8 }

// ===== Key構造体 (attack6で使用) =====
#[derive(Clone, Copy)]
struct Key { key0: u32, key1: u32, key2: u32 }

impl Key {
    fn new(k0: u32, k1: u32, k2: u32) -> Self { Key { key0: k0, key1: k1, key2: k2 } }
    fn update(&mut self, c: u8) {
        self.key0 = crc32(self.key0, c);
        self.key1 = (self.key1.wrapping_add(self.key0 & 0xff)).wrapping_mul(0x08088405).wrapping_add(1);
        self.key2 = crc32(self.key2, (self.key1 >> 24) as u8);
    }
    fn decrypt_byte(&self) -> u8 { decrypt_byte(self.key2) }
}

// ===== グローバル状態をまとめた構造体 =====
struct State {
    r: [[u8; 10]; 7],
    c: [[u8; 12]; 7],
    // attack0/1で求める中間値
    key0_crc:  u32,
    key1_mul1: u32,
    key1_mul2: u32,
    key2_crc:  u32,
    s0: [[u8; 2]; 7],
    // attack1/3で求める中間値
    key0_10lsb: [u8; 7],
    key0_11lsb: [u8; 7],
    key1_10msb: [u8; 7],
    key1_11msb: [u8; 7],
    key0_20lsb: [u8; 7],
    key0_21lsb: [u8; 7],
    key1_20msb: [u8; 7],
    key1_21msb: [u8; 7],
    // 最終的な鍵
    key0: u32,
    key1: u32,
    key2: u32,
}

impl State {
    fn new(r: [[u8; 10]; 7], c: [[u8; 12]; 7]) -> Self {
        State {
            r, c,
            key0_crc: 0, key1_mul1: 0, key1_mul2: 0, key2_crc: 0,
            s0: [[0; 2]; 7],
            key0_10lsb: [0; 7], key0_11lsb: [0; 7],
            key1_10msb: [0; 7], key1_11msb: [0; 7],
            key0_20lsb: [0; 7], key0_21lsb: [0; 7],
            key1_20msb: [0; 7], key1_21msb: [0; 7],
            key0: 0, key1: 0, key2: 0,
        }
    }

    // ===== attack6: key0の上位バイトを総当たりし、key0/key1/key2を検証 =====
    fn attack6(&mut self) -> bool {
        for g0 in 0..0x100u32 {
            for g1 in 0..0x100u32 {
                let key0_h = (g1 << 16) | self.key0_crc;
                let key0 = ((key0_h ^ CRC_TABLE[g0 as usize]) << 8) | g0;

                let mut keys0 = Key::new(key0, self.key1, self.key2);
                let mut keys1 = keys0;
                let mut ok = true;
                for i in 0..10 {
                    // keys1 は1バイト先行 (C コードの keys1 = keys0 後に keys0.update→keys1.update の順)
                    let t = self.c[0][i] ^ keys1.decrypt_byte();
                    keys1.update(t);
                    let t2 = t ^ keys0.decrypt_byte();
                    keys0.update(t2);
                    if t2 != self.r[0][i] { ok = false; break; }
                }
                if ok {
                    self.key0 = key0;
                    return true;
                }
            }
        }
        false
    }

    // ===== attack5: key1を24ビット総当たり =====
    fn attack5(&mut self) -> bool {
        for g12 in 0..0x1000000u32 {
            let key1 = (self.key1_mul1 | g12).wrapping_mul(0xd94fa8cd);
            if msb(key1.wrapping_mul(0xd4652819)) == msb(self.key1_mul2) {
                let mut ok = true;
                for f in 0..7 {
                    let k1 = (key1.wrapping_add(self.key0_10lsb[f] as u32))
                        .wrapping_mul(0x08088405).wrapping_add(1);
                    if msb(k1) != self.key1_10msb[f] { ok = false; break; }
                    let k1 = (k1.wrapping_add(self.key0_20lsb[f] as u32))
                        .wrapping_mul(0x08088405).wrapping_add(1);
                    if msb(k1) != self.key1_20msb[f] { ok = false; break; }
                    let k1 = (key1.wrapping_add(self.key0_11lsb[f] as u32))
                        .wrapping_mul(0x08088405).wrapping_add(1);
                    if msb(k1) != self.key1_11msb[f] { ok = false; break; }
                    let k1 = (k1.wrapping_add(self.key0_21lsb[f] as u32))
                        .wrapping_mul(0x08088405).wrapping_add(1);
                    if msb(k1) != self.key1_21msb[f] { ok = false; break; }
                }
                if ok {
                    self.key1 = key1;
                    if self.attack6() { return true; }
                }
            }
        }
        false
    }

    // ===== attack4: key2の下位バイトを総当たり =====
    fn attack4(&mut self) -> bool {
        eprintln!("  attack4 {:08x} {:08x} {:08x}", self.key0_crc, self.key1_mul2, self.key2_crc);
        for g11 in 0..0x100u32 {
            let key2 = ((self.key2_crc ^ CRC_TABLE[g11 as usize]) << 8) | g11;
            if decrypt_byte(key2) == self.s0[0][0] {
                self.key2 = key2;
                if self.attack5() { return true; }
            }
        }
        false
    }

    // ===== attack3: 7ファイル分の中間値を再帰的に絞り込む =====
    fn attack3(&mut self, f: usize) -> bool {
        if f == 7 { return self.attack4(); }

        for g0 in 0..2u8 {
            for g1 in 0..2u8 {
                // --- position 0→2 (keystream index 2) ---
                self.key0_20lsb[f] = (crc32(self.key0_crc ^ CRC_TABLE[self.r[f][0] as usize], self.r[f][1]) & 0xff) as u8;
                self.key1_20msb[f] = msb(self.key1_mul2)
                    .wrapping_add(0x08)
                    .wrapping_add(g0)
                    .wrapping_add(msb(
                        (self.key0_10lsb[f] as u32).wrapping_mul(0xd4652819)
                        .wrapping_add((self.key0_20lsb[f] as u32).wrapping_mul(0x08088405))
                    ));
                let k20 = crc32(self.key2_crc ^ CRC_TABLE[self.key1_10msb[f] as usize], self.key1_20msb[f]);
                let s20 = decrypt_byte(k20);

                // --- position 1→2 (keystream index 2, second interleave) ---
                self.key0_21lsb[f] = (crc32(
                    self.key0_crc ^ CRC_TABLE[(self.r[f][0] ^ self.s0[0][0]) as usize],
                    self.r[f][1] ^ self.s0[f][1],
                ) & 0xff) as u8;
                self.key1_21msb[f] = msb(self.key1_mul2)
                    .wrapping_add(0x08)
                    .wrapping_add(g1)
                    .wrapping_add(msb(
                        (self.key0_11lsb[f] as u32).wrapping_mul(0xd4652819)
                        .wrapping_add((self.key0_21lsb[f] as u32).wrapping_mul(0x08088405))
                    ));
                let k21 = crc32(self.key2_crc ^ CRC_TABLE[self.key1_11msb[f] as usize], self.key1_21msb[f]);
                let s21 = decrypt_byte(k21);

                if (self.r[f][2] ^ s20 ^ s21) == self.c[f][2] {
                    if self.attack3(f + 1) { return true; }
                }
            }
        }
        false
    }

    // ===== attack2: 残りの中間値ビットを総当たり =====
    fn attack2(&mut self) -> bool {
        eprintln!("attack2 {:08x} {:08x} {:08x}", self.key0_crc, self.key1_mul1, self.key2_crc);

        // key0_crc の bits[15:8] を総当たり
        for g0 in 0..0x100u32 {
            self.key0_crc = (self.key0_crc & !0xff00) | (g0 << 8);
            // key1_mul2 の MSB を総当たり
            for g1 in 0..0x100u32 {
                self.key1_mul2 = g1 << 24;
                // key2_crc の bits[23:16] と bits[1:0] を総当たり
                for g2 in 0..0x100u32 {
                    for g3 in 0..4u32 {
                        self.key2_crc = (g2 << 16) | (self.key2_crc & !0xff0003) | g3;
                        if self.attack3(0) { return true; }
                    }
                }
            }
        }
        false
    }

    // ===== attack1: 7ファイル分のkeystream MSBを絞り込む (再帰) =====
    fn attack1(&mut self, f: usize) -> bool {
        if f == 7 { return self.attack2(); }

        for g0 in 0..2u8 {
            for g1 in 0..2u8 {
                // keystream index 1 (position 0 → 1)
                self.key0_10lsb[f] = (self.key0_crc ^ CRC_TABLE[self.r[f][0] as usize]) as u8;
                self.key1_10msb[f] = msb(self.key1_mul1)
                    .wrapping_add(msb((self.key0_10lsb[f] as u32).wrapping_mul(0x08088405)))
                    .wrapping_add(g0);
                self.s0[f][1] = decrypt_byte(CRC_TABLE[self.key1_10msb[f] as usize] ^ self.key2_crc);

                // keystream index 1 (interleave 2)
                self.key0_11lsb[f] = (self.key0_crc ^ CRC_TABLE[(self.r[f][0] ^ self.s0[0][0]) as usize]) as u8;
                self.key1_11msb[f] = msb(self.key1_mul1)
                    .wrapping_add(msb((self.key0_11lsb[f] as u32).wrapping_mul(0x08088405)))
                    .wrapping_add(g1);
                let s1_1 = decrypt_byte(CRC_TABLE[self.key1_11msb[f] as usize] ^ self.key2_crc);

                if (self.r[f][1] ^ self.s0[f][1] ^ s1_1) == self.c[f][1] {
                    if self.attack1(f + 1) { return true; }
                }
            }
        }
        false
    }

    // ===== attack0: 最外ループ =====
    fn attack0(&mut self) -> bool {
        for g0 in 0..0x100u32 {         // key0_crc の LSB
            self.key0_crc = g0;
            for g1 in 0..0x100u32 {     // key1_mul1 の MSB
                self.key1_mul1 = g1 << 24;
                for g2 in 0..0x4000u32 { // key2_crc の bits[15:2]
                    self.key2_crc = g2 << 2;
                    for g3 in 0..=255u8 { // s0[0][0]: R[0][0]^P[0][0]
                        self.s0[0][0] = g3;
                        if self.attack1(0) { return true; }
                    }
                }
            }
        }
        false
    }
}

fn main() {
    // ===== flag.zip からCデータを読み込む =====
    let pos: [u64; 7] = [0x43, 0x141, 0x23f, 0x33d, 0x43b, 0x539, 0x637];
    let mut c_data = [[0u8; 12]; 7];
    let mut f_file = File::open("flag.zip").expect("flag.zip not found");
    for i in 0..7 {
        f_file.seek(SeekFrom::Start(pos[i])).unwrap();
        f_file.read_exact(&mut c_data[i]).unwrap();
    }

    // ===== 既知平文 (guessRandで求めた値) =====
    // C コードの R_[] に対応 (guessRand の結果として確認済み)
    let r_data: [[u8; 10]; 7] = [
        [144,  89,  71, 232, 134, 190,  63, 187,  99, 153],
        [207, 209, 189, 126, 170, 203,  34, 140, 192,   1],
        [112,  51,  10, 162, 191,  75, 138,  38, 160,  35],
        [107,  49, 124, 178,  25,   3, 113,  89, 191, 212],
        [242, 142, 166, 176,  13,  80, 123,  48, 221,  60],
        [ 49,  77, 111,  59, 240,  47, 135, 122,  85,  40],
        [158, 192,  89,  27, 115, 115,  31, 228, 205, 222],
    ];

    println!("Starting attack...");
    let mut state = State::new(r_data, c_data);

    if state.attack0() {
        println!("key0: {:08x}", state.key0);
        println!("key1: {:08x}", state.key1);
        println!("key2: {:08x}", state.key2);
    } else {
        println!("Attack failed.");
    }
}

main();

Starting attack...


attack2 00000008 00000000 00002ce8
attack2 0000ff08 00000000 0000ace8
attack2 0000ff08 80000000 00002fc8
attack2 0000ff08 80000000 0000afc8
attack2 00000009 00000000 000024d8
attack2 0000ff09 00000000 0000a4d8
attack2 0000ff09 80000000 000027f8
attack2 0000ff09 80000000 0000a7f8
attack2 0000000a 00000000 00003c8c
attack2 0000ff0a 00000000 0000bc8c
attack2 0000ff0a 80000000 00003fac
attack2 0000ff0a 80000000 0000bfac
attack2 0000000b 00000000 000034bc
attack2 0000ff0b 00000000 0000b4bc
attack2 0000ff0b 80000000 0000379c
attack2 0000ff0b 80000000 0000b79c
attack2 00000010 00000000 000002b8
attack2 0000ff10 00000000 00006e58
attack2 0000ff10 00000000 000082b8
attack2 0000ff10 00000000 0000ee58
attack2 0000ff10 80000000 00000198
attack2 0000ff10 80000000 00006d78
attack2 0000ff10 80000000 00008198
attack2 0000ff10 80000000 0000ed78
attack2 00000011 00000000 00000a88
attack2 0000ff11 00000000 00006668
attack2 0000ff11 00000000 00008a88
attack2 0000ff11 00000000 0000e668
attack2 0000ff11 800

attack2 0000ff93 00000000 0000def4
attack2 0000ff93 80000000 00003134
attack2 0000ff93 80000000 00005dd4
attack2 0000ff93 80000000 0000b134
attack2 0000ff93 80000000 0000ddd4
attack2 00000094 00000000 00000a88
attack2 0000ff94 00000000 00006668
attack2 0000ff94 00000000 00008a88
attack2 0000ff94 00000000 0000e668
attack2 0000ff94 80000000 000009a8
attack2 0000ff94 80000000 00006548
attack2 0000ff94 80000000 000089a8
attack2 0000ff94 80000000 0000e548
attack2 00000095 00000000 000002b8
attack2 0000ff95 00000000 00006e58
attack2 0000ff95 00000000 000082b8
attack2 0000ff95 00000000 0000ee58
attack2 0000ff95 80000000 00000198
attack2 0000ff95 80000000 00006d78
attack2 0000ff95 80000000 00008198
attack2 0000ff95 80000000 0000ed78
attack2 00000096 00000000 00001aec
attack2 0000ff96 00000000 0000760c
attack2 0000ff96 00000000 00009aec
attack2 0000ff96 00000000 0000f60c
attack2 0000ff96 80000000 000019cc
attack2 0000ff96 80000000 0000752c
attack2 0000ff96 80000000 000099cc
attack2 0000ff96 800

key0: d0df8cc7
key1: c0187ed5
key2: 2201a0cd


```bash
bkcrack -C flag.zip -k d0df8cc7 c0187ed5 2201a0cd -d flag_decrypted.zip
```

In [6]:
use std::fs;

fn main() {
    let dir = "./flag_decrypted";
    
    // 1枚目を読み込んでベースにする
    let mut result = fs::read(format!("{}/flag1.txt", dir)).expect("flag1.txt not found");

    // 2〜7枚目をXOR
    for i in 2..=7 {
        let data = fs::read(format!("{}/flag{}.txt", dir, i))
            .expect(&format!("flag{}.txt not found", i));
        for (r, d) in result.iter_mut().zip(data.iter()) {
            *r ^= d;
        }
    }

    fs::write("flag.txt", result);
}

main()

()